# HSH targets: DP2 light curves and reports

Batch example for the observed HSH inventory. It imports observation epochs, uses curated target coordinates, queries DP2 coverage and DIA photometry, and produces light curves and per-target reports. MOP and Rubin caches are shared across runs. TAP and Butler steps require Rubin RSP access and may be expensive.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "target_selection" / "cache_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró la raíz del repositorio target_selection.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from target_selection.cache_paths import OUTPUT_ROOT, CACHE_ROOT
from target_selection import (load_target_list, query_lsst_coverage,
                              compute_lsst_photometry, create_lightcurve_plots,
                              create_target_reports)

DATA_RELEASE = "DP2"
MAX_WORKERS = 4
OUTPUT_DIR = OUTPUT_ROOT / "notebooks" / "hsh_dp2_lightcurves_and_reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. HSH targets: MOP + HSH epochs + DP2 DIA curves

This optional batch task uses every target present in the HSH inventory. It queries DP2 coverage and retrieves both published DIA products: `ForcedSourceOnDiaObject` and `DiaSource`. The two measurements are plotted with different markers. HSH epochs are read from the local registry, while MOP photometry is loaded from the persistent MOP cache when available.


In [5]:
# This cell is intentionally not executed automatically: it can issue many TAP queries.
from target_selection.target_registry import TargetRegistry
from target_selection.tasks.surveys import import_hsh_inventory, observed_targets

# Use the repository root found in the setup cell, regardless of Jupyter cwd.
HSH_INVENTORY = PROJECT_ROOT / "data" / "hsh_data" / "image_collection_astro.csv"
RUN_OUTPUT_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else PROJECT_ROOT / OUTPUT_DIR
HSH_DATABASE = OUTPUT_ROOT / "target_database" / "target_selection.sqlite"
HSH_LIGHTCURVE_DIR = RUN_OUTPUT_DIR / "hsh_lightcurves"

# Import/update local HSH observation epochs, then obtain all observed HSH targets.
import_hsh_inventory(HSH_INVENTORY, database_path=HSH_DATABASE, force=False)
hsh_targets = observed_targets(providers=("HSH",), database_path=HSH_DATABASE)
# The HSH inventory identifies observed objects but its image CRVAL values are
# pointings, not event coordinates. Complete positions from the curated HSH list.
hsh_coordinate_targets = load_target_list(
    PROJECT_ROOT / "data" / "hsh_data" / "hsh_objects.csv",
    source_name="HSH",
    column_map={"Target": "object", "RA_deg": "ra_deg", "Dec_deg": "dec_deg"},
)
hsh_targets = hsh_targets.drop(columns=["RA_deg", "Dec_deg"], errors="ignore").merge(
    hsh_coordinate_targets[["Target", "RA_deg", "Dec_deg"]],
    on="Target", how="left", validate="one_to_one",
)
missing_hsh_coordinates = hsh_targets["RA_deg"].isna() | hsh_targets["Dec_deg"].isna()
print(f"HSH targets selected: {len(hsh_targets)}; missing coordinates: {int(missing_hsh_coordinates.sum())}")

# Query DP2 visit coverage once for all HSH targets.
hsh_coverage = query_lsst_coverage(
    hsh_targets, data_release=DATA_RELEASE, max_workers=MAX_WORKERS,
    output_path=RUN_OUTPUT_DIR / "hsh_dp2_coverage.csv",
)

# Retrieve both DiaSource and ForcedSourceOnDiaObject products.
hsh_dia_photometry = compute_lsst_photometry(
    hsh_targets, coverage=hsh_coverage, data_release=DATA_RELEASE,
    method="dia_combined_catalog", max_workers=MAX_WORKERS,
    output_path=RUN_OUTPUT_DIR / "hsh_dp2_dia_photometry.csv",
    persistent_target_dir=CACHE_ROOT / "rubin" / "photometry",
)

# Load HSH epochs from the same local registry.
hsh_epochs = TargetRegistry(HSH_DATABASE).observation_epochs(hsh_targets, providers=("HSH",))

# Persistent PNGs: MOP points (if cached), HSH coverage, and both DP2 curves.
hsh_lightcurve_pngs = create_lightcurve_plots(
    hsh_targets, output_dir=HSH_LIGHTCURVE_DIR,
    mop_photometry_dir=CACHE_ROOT / "mop" / "photometry",
    release_photometry=hsh_dia_photometry, coverage=hsh_coverage,
    observatory_epochs=hsh_epochs, data_release=DATA_RELEASE,
    layers=("mop_photometry", "release_photometry", "release_epochs", "hsh"),
    overwrite=False, verbose=True,
)
print(f"HSH light-curve plots: {len(hsh_lightcurve_pngs)}")


[surveys] Importing HSH inventory: /home/karennowo/target_selection/hsh_data/image_collection_astro.csv
[surveys] HSH records imported/updated: 0
[surveys] Previously observed targets: 29
[targets] Loading HSH targets from /home/karennowo/target_selection/hsh_data/hsh_objects.csv.
[targets] Normalized targets: 49
HSH targets selected: 29; missing coordinates: 0
[lsst] Querying DP2 coverage for 29 targets.
[lsst] Coverage rows: 2586 (24 targets).
[lsst] DiaSource rows returned: 159 (142 measured; 29 targets).
[lsst] Photometry rows returned: 989 using dia_combined_catalog.
[lsst] Saved 989 photometry rows to /home/karennowo/target_selection/outputs/task_api_example/hsh_dp2_dia_photometry.csv.


/home/karennowo/target_selection/target_selection/sources/lsst.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty, ignore_index=True, sort=False) if nonempty else pd.DataFrame()


[reports] Updating 29 light-curve PNG(s).


Light curves:   0%|          | 0/29 [00:00<?, ?target/s]

[reports] Light-curve PNGs ready: 29 in /home/karennowo/target_selection/outputs/task_api_example/hsh_lightcurves
HSH light-curve plots: 29


## 2. HSH target reports without visibility selection

This task is independent of visibility. It creates one dashboard per HSH-observed target, using the DP2 coverage and DIA photometry already computed above. Targets without a covering DP2 coadd are reported and skipped.


In [8]:
# Uses hsh_targets, hsh_coverage, and hsh_dia_photometry from section 1.
hsh_target_reports = create_target_reports(
    hsh_targets,
    coverage=hsh_coverage,
    data_release=DATA_RELEASE,
    release_photometry=hsh_dia_photometry,
    output_dir=RUN_OUTPUT_DIR / "target_reports",
    mop_photometry_dir=CACHE_ROOT / "mop" / "photometry",
    max_workers=MAX_WORKERS,
    overwrite=True,
    verbose=True,
)
print(f"HSH target reports generated or reused: {len(hsh_target_reports)}")


[reports] Creating 29 target report(s).
[reports] Target reports are generated sequentially; max_workers is ignored.


Target reports:   0%|          | 0/29 [00:00<?, ?target/s]

[reports] Building target report: ASASSN-24fs
[reports] Report saved: /home/karennowo/target_selection/outputs/task_api_example/target_reports/ASASSN-24fs_target_report.png
[reports] Building target report: Gaia23cnu
Photometry skipped for Gaia23cnu: no MOP client or cached photometry
[reports] Report saved: /home/karennowo/target_selection/outputs/task_api_example/target_reports/Gaia23cnu_target_report.png
[reports] Building target report: Gaia23dpn
Photometry skipped for Gaia23dpn: no MOP client or cached photometry
[reports] Skipping Gaia23dpn: No coadd covers target 'Gaia23dpn'
[reports] Building target report: Gaia24bsi
[reports] Skipping Gaia24bsi: No coadd covers target 'Gaia24bsi'
[reports] Building target report: Gaia24bsj
[reports] Skipping Gaia24bsj: No coadd covers target 'Gaia24bsj'
[reports] Building target report: Gaia24ddl
[reports] Skipping Gaia24ddl: No coadd covers target 'Gaia24ddl'
[reports] Building target report: OGLE-2023-BLG-0723
[reports] Report saved: /home/k